# Notebook 4: Model Training
**Project:** Can FDA Drug Approvals Predict Stock Price Movements?  
**Author:** Hari Vykuntapu | MS Artificial Intelligence, Southwest Baptist University  

---

The framing is binary classification: given an FDA approval event with its full feature vector (including RSS, FinBERT sentiment, VADER sentiment, and structural metadata), predict whether the stock is UP or DOWN seven trading days later.

I'm training three models at increasing complexity:
1. **Logistic Regression** — interpretable linear baseline
2. **Random Forest** — non-linear ensemble, captures feature interactions
3. **XGBoost** — gradient boosting, often best in tabular settings

The goal isn't just accuracy — it's understanding *which features* drive the prediction. SHAP values in the next notebook will decompose each model's decisions.

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)
from sklearn.pipeline import Pipeline

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print('XGBoost not installed — will skip XGB model.')

os.makedirs('../models', exist_ok=True)
os.makedirs('../outputs/results', exist_ok=True)

RANDOM_STATE = 42
print('Setup complete.')

Setup complete.


## Load Features

In [2]:
df = pd.read_csv('../data/processed/fda_features.csv')
print(f'Feature dataset: {df.shape}')
print(f'Target balance: {df["price_up_7d"].value_counts().to_dict()}')
df.head(3)

Feature dataset: (296, 23)
Target balance: {1.0: 160, 0.0: 136}


,ticker,drug_name,approval_date,approval_year,app_type_clean,approval_type_weight,drug_novelty_score,market_timing_factor,rss_score,finbert_compound,...,vader_pos,vader_neg,day_of_week,market_cap_category,prior_approvals_count,approval_quarter,return_1d,return_3d,return_7d,price_up_7d
0,LLY,Vellizumab,2021-07-18,2021,NDA,1.0,0.5,0.8,0.785,0.0,...,0.171,0.0,6,3,14,3,0.0000,1.3453,3.8737,1.0
1,AMGN,Vellizumab,2018-11-27,2018,ANDA,0.3,0.0,1.0,0.335,0.0,...,0.256,0.0,1,2,4,4,2.1918,5.6571,0.3349,1.0
2,ABBV,Duninib,2022-08-13,2022,ANDA,0.3,0.0,0.8,0.295,0.0,...,0.181,0.0,5,3,19,3,0.0000,0.1827,-1.3704,0.0


## Feature Selection\n\nTen features across three groups:\n\n- **Regulatory:** `rss_score`, `approval_type_weight`, `drug_novelty_score`, `market_timing_factor` — the structured metadata features. RSS is a composite of the other three, but I include all four because the model may weight the interaction differently than my fixed formula does.\n- **NLP:** `finbert_compound`, `vader_compound` — the text-derived sentiment scores.\n- **Structural:** `market_cap_category`, `day_of_week`, `approval_quarter`, `prior_approvals_count` — context features that give the model information about the event environment.\n\nRaw return columns are excluded (that's leakage). The application type string is excluded because it's already encoded numerically in `approval_type_weight`.

In [3]:
FEATURE_COLS = [
    'rss_score',
    'approval_type_weight',
    'drug_novelty_score',
    'market_timing_factor',
    'finbert_compound',
    'vader_compound',
    'market_cap_category',
    'day_of_week',
    'approval_quarter',
    'prior_approvals_count',
]

TARGET = 'price_up_7d'

# Only use columns that exist in the dataframe
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

X = df[FEATURE_COLS].copy()
y = df[TARGET].astype(int)

# Fill any remaining NaN with column medians
X = X.fillna(X.median())

print(f'Features: {FEATURE_COLS}')
print(f'X shape: {X.shape}, y shape: {y.shape}')
print(f'Class balance — UP: {y.sum()}, DOWN: {(y==0).sum()}')

Features: ['rss_score', 'approval_type_weight', 'drug_novelty_score', 'market_timing_factor', 'finbert_compound', 'vader_compound', 'market_cap_category', 'day_of_week', 'approval_quarter', 'prior_approvals_count']
X shape: (296, 10), y shape: (296,)
Class balance — UP: 160, DOWN: 136


## Train/Test Split — 80/20 Stratified

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Train class balance — UP: {y_train.sum()}, DOWN: {(y_train==0).sum()}')
print(f'Test class balance  — UP: {y_test.sum()}, DOWN: {(y_test==0).sum()}')

Train: 236 samples | Test: 60 samples
Train class balance — UP: 128, DOWN: 108
Test class balance  — UP: 32, DOWN: 28


## Model 1: Logistic Regression (Baseline)

In [5]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, C=1.0))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]

lr_metrics = {
    'model': 'Logistic Regression',
    'accuracy': round(accuracy_score(y_test, y_pred_lr), 4),
    'precision': round(precision_score(y_test, y_pred_lr, zero_division=0), 4),
    'recall': round(recall_score(y_test, y_pred_lr, zero_division=0), 4),
    'f1': round(f1_score(y_test, y_pred_lr, zero_division=0), 4),
    'auroc': round(roc_auc_score(y_test, y_prob_lr), 4),
}
print('Logistic Regression results:')
for k, v in lr_metrics.items():
    print(f'  {k}: {v}')

Logistic Regression results:
  model: Logistic Regression
  accuracy: 0.5333
  precision: 0.5455
  recall: 0.75
  f1: 0.6316
  auroc: 0.4431


## Model 2: Random Forest

In [6]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

rf_metrics = {
    'model': 'Random Forest',
    'accuracy': round(accuracy_score(y_test, y_pred_rf), 4),
    'precision': round(precision_score(y_test, y_pred_rf, zero_division=0), 4),
    'recall': round(recall_score(y_test, y_pred_rf, zero_division=0), 4),
    'f1': round(f1_score(y_test, y_pred_rf, zero_division=0), 4),
    'auroc': round(roc_auc_score(y_test, y_prob_rf), 4),
}
print('Random Forest results:')
for k, v in rf_metrics.items():
    print(f'  {k}: {v}')

Random Forest results:
  model: Random Forest
  accuracy: 0.5
  precision: 0.5263
  recall: 0.625
  f1: 0.5714
  auroc: 0.5525


## Model 3: XGBoost

In [7]:
if XGB_AVAILABLE:
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        scale_pos_weight=(y_train==0).sum() / y_train.sum(),
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        verbosity=0,
    )

    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred_xgb = xgb_model.predict(X_test)
    y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

    xgb_metrics = {
        'model': 'XGBoost',
        'accuracy': round(accuracy_score(y_test, y_pred_xgb), 4),
        'precision': round(precision_score(y_test, y_pred_xgb, zero_division=0), 4),
        'recall': round(recall_score(y_test, y_pred_xgb, zero_division=0), 4),
        'f1': round(f1_score(y_test, y_pred_xgb, zero_division=0), 4),
        'auroc': round(roc_auc_score(y_test, y_prob_xgb), 4),
    }
    print('XGBoost results:')
    for k, v in xgb_metrics.items():
        print(f'  {k}: {v}')
else:
    xgb_metrics = None
    print('XGBoost skipped.')

XGBoost results:
  model: XGBoost
  accuracy: 0.5333
  precision: 0.5625
  recall: 0.5625
  f1: 0.5625
  auroc: 0.5725


## Model Comparison & Best Model Selection

In [8]:
all_metrics = [lr_metrics, rf_metrics]
if xgb_metrics:
    all_metrics.append(xgb_metrics)

metrics_df = pd.DataFrame(all_metrics).set_index('model')
print('=== Model Comparison ===')
print(metrics_df.to_string())

best_model_name = metrics_df['auroc'].idxmax()
print(f'\nBest model by AUROC: {best_model_name}')

model_map = {
    'Logistic Regression': lr_pipeline,
    'Random Forest': rf_model,
    'XGBoost': xgb_model if XGB_AVAILABLE else None,
}
best_model = model_map[best_model_name]

# Save best model
joblib.dump(best_model, '../models/best_model.pkl')
print(f'Saved best model to models/best_model.pkl')

# Save all models
joblib.dump(lr_pipeline, '../models/logistic_regression.pkl')
joblib.dump(rf_model, '../models/random_forest.pkl')
if XGB_AVAILABLE:
    joblib.dump(xgb_model, '../models/xgboost.pkl')

# Save metrics
metrics_df.to_csv('../outputs/results/model_metrics.csv')

# Save model metadata
metadata = {
    'best_model': best_model_name,
    'feature_cols': FEATURE_COLS,
    'target': TARGET,
    'train_size': int(X_train.shape[0]),
    'test_size': int(X_test.shape[0]),
    'metrics': metrics_df.to_dict(),
}
with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('All models and metadata saved.')

=== Model Comparison ===
                     accuracy  precision  recall      f1   auroc
model                                                           
Logistic Regression    0.5333     0.5455  0.7500  0.6316  0.4431
Random Forest          0.5000     0.5263  0.6250  0.5714  0.5525
XGBoost                0.5333     0.5625  0.5625  0.5625  0.5725

Best model by AUROC: XGBoost
Saved best model to models/best_model.pkl


All models and metadata saved.


## Cross-Validation Check

A single 80/20 split is a bit fragile with 296 samples. I run 5-fold stratified CV on the best model to check whether the test performance is reproducible or just lucky. Moderate variance in CV AUROC (σ around 0.04–0.06) would be normal for a dataset this size; large variance would mean the split is doing the heavy lifting.

In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores = cross_val_score(best_model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'5-Fold CV AUROC ({best_model_name}):')
print(f'  Scores: {[round(s, 4) for s in cv_scores]}')
print(f'  Mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

5-Fold CV AUROC (XGBoost):
  Scores: [np.float64(0.5837), np.float64(0.6701), np.float64(0.625), np.float64(0.5868), np.float64(0.5891)]
  Mean: 0.6110 ± 0.0332
